# Project 5: Classification Demo — K-NN & Decision Tree

**Dataset:** California Housing (binned into High/Low price)

**Goal:** Introduction to classification algorithms. Turn a regression problem into a binary classification problem and compare K-NN vs Decision Tree.

In [ ]:
# ====== Setup: imports + data loader ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120

DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    """Load a real dataset from GitHub."""
    df = pd.read_csv(f"{DATA_URL}/{fn}")
    print(f"Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols")
    return df
print('Setup OK!')
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Turn Regression Into Classification

In [ ]:
# Convert continuous house value into binary categories
# 将连续房价转为二分类: 高于中位数=High, 低于=Low
df = load_data('housing.csv')
median = df['MedHouseVal'].median()
df['PriceCat'] = (df['MedHouseVal'] > median).astype(int)

print(f'Median price: ${median*100:.0f}K')
print(f'Low (0):  {(df.PriceCat==0).sum():,} ({(df.PriceCat==0).mean():.1%})')
print(f'High (1): {(df.PriceCat==1).sum():,} ({(df.PriceCat==1).mean():.1%})')

features = ['MedInc','HouseAge','AveRooms','AveBedrms',
            'Population','AveOccup','Latitude','Longitude']
X = df[features]; y = df['PriceCat']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print(f'Train: {len(X_train):,}  Test: {len(X_test):,}')

## 2. K-Nearest Neighbours (K-NN)

K-NN works by finding the K nearest data points and letting them vote. The key hyperparameter is **k** — how many neighbors to consult. Too small = overfitting, too large = underfitting.

In [ ]:
# Try different k values — 尝试不同的K值
print('K-NN Accuracy by k:')
for k in [1, 3, 5, 7, 11, 15, 21, 31, 51, 101]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_s, y_train)
    acc = knn.score(X_test_s, y_test)
    print(f'  k={k:3d}: Accuracy={acc:.4f}')

print()
print('Key insight:')
print('  k=1: too sensitive to noise (overfitting)')
print('  k=7: sweet spot (best accuracy ~83%)')
print('  k=101: too smooth (underfitting, ~79%)')

## 3. Decision Tree

Decision Trees are **interpretable** — you can trace every prediction back to a specific if-then rule. This transparency is crucial for regulated industries (banking, healthcare).

In [ ]:
# Decision Tree with depth limit to prevent overfitting
# 限制深度的决策树 — 防止过拟合，提高可解释性
dt = DecisionTreeClassifier(max_depth=5, random_state=42, min_samples_leaf=10)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_test)

print(f'Decision Tree Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Low','High']))

# Feature importance — 特征重要性 (决策树自动计算!)
fi = pd.DataFrame({'Feature': features, 'Importance': dt.feature_importances_})
fi = fi.sort_values('Importance', ascending=False)
print('\nFeature Importance (decision tree view):')
print(fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fi['Feature'], fi['Importance'], color='#2e86de')
ax.set_title('Decision Tree Feature Importance')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

## 4. Model Comparison

Both models solve the same problem, but they make different trade-offs.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

knn_best = KNeighborsClassifier(n_neighbors=7).fit(X_train_s, y_train)
y_knn = knn_best.predict(X_test_s)
y_dt = DecisionTreeClassifier(max_depth=5, random_state=42,
    min_samples_leaf=10).fit(X_train, y_train).predict(X_test)

comp = pd.DataFrame({
    'Metric': ['Accuracy','Precision (High)','Recall (High)','F1 (High)'],
    'K-NN (k=7)': [
        accuracy_score(y_test, y_knn),
        precision_score(y_test, y_knn),
        recall_score(y_test, y_knn),
        f1_score(y_test, y_knn)],
    'Decision Tree': [
        accuracy_score(y_test, y_dt),
        precision_score(y_test, y_dt),
        recall_score(y_test, y_dt),
        f1_score(y_test, y_dt)],
}).round(4)
print(comp.to_string(index=False))
print()
print('Trade-off: K-NN is more accurate, Decision Tree is explainable.')
print('Which one you choose depends on your use case.')

## Check Your Understanding

1. **k=1 vs k=7:** Why does k=1 give lower accuracy than k=7? What problem does k=1 suffer from?
2. **Interpretability:** Which model can explain *why* it made a specific prediction? Give an example situation where this matters.
3. **Feature importance:** Decision Tree says MedInc is the most important feature. Is this expected? Why?
4. **Recall gap:** Decision Tree's recall for "High" is lower than for "Low". What does this mean in practice?